#BIKRAM SHAH(ACE079BCT019)
# Data Mining Lab Assignment
# Title: Cluster Analysis on Restaurant Orders Dataset (Customer Segmentation)

This report performs cluster analysis on customers derived from the restaurant/food-delivery order dataset, comparing K-means, hierarchical clustering, and DBSCAN with evaluation and scalability discussion.

Since the raw dataset (`Customer_ID`, `Date`, `Product`) is transactional rather than a table of numeric features, customer-level behavioural features are first engineered (order volume, product variety, basket size, activity span) so the same clustering pipeline used on the Iris dataset can be applied here to segment customers.

## Objectives
- Load the restaurant orders dataset and engineer customer-level features.
- Apply K-means clustering, hierarchical clustering, and DBSCAN.
- Compare algorithm performance using internal evaluation metrics.
- Visualize cluster structure and discuss issues such as scalability and model selection.

In [ ]:
import sys
!{sys.executable} -m pip install pandas matplotlib scikit-learn scipy --quiet

In [ ]:
import time
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans, AgglomerativeClustering, DBSCAN
from sklearn.metrics import silhouette_score, davies_bouldin_score
from scipy.cluster.hierarchy import dendrogram, linkage

DATA_PATH = Path('restaurant_orders.csv')
assert DATA_PATH.exists(), f'Dataset not found: {DATA_PATH}'
raw = pd.read_csv(DATA_PATH)
raw['Date'] = pd.to_datetime(raw['Date'])
raw['transaction_id'] = raw['Customer_ID'].astype(str) + '_' + raw['Date'].dt.strftime('%Y%m%d')

# Engineer customer-level behavioural features from the order log
customers = raw.groupby('Customer_ID').agg(
    total_purchases=('Product', 'count'),
    unique_products=('Product', pd.Series.nunique),
    num_transactions=('transaction_id', pd.Series.nunique),
    active_days=('Date', lambda x: (x.max() - x.min()).days + 1),
).reset_index()
customers['avg_basket_size'] = customers['total_purchases'] / customers['num_transactions']
customers['product_variety_ratio'] = customers['unique_products'] / customers['total_purchases']

feature_names = ['total_purchases', 'unique_products', 'num_transactions', 'active_days', 'avg_basket_size', 'product_variety_ratio']
X = customers[feature_names]

print('Dataset shape:', X.shape)
print('Feature names:', feature_names)
customers.head()

Each row now represents one customer, described by six behavioural features derived from their order history: total items ordered, product variety, number of orders, days active, average basket size, and product variety ratio. This mirrors the numeric feature table that the Iris example clusters directly.

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_scaled)

print('PCA explained variance ratio:', np.round(pca.explained_variance_ratio_, 3))

fig, ax = plt.subplots(figsize=(6, 4))
ax.scatter(X_pca[:, 0], X_pca[:, 1], c='steelblue', edgecolor='k', s=50)
ax.set_title('Customer features PCA projection')
ax.set_xlabel('PCA 1')
ax.set_ylabel('PCA 2')
plt.show()

The customer features are standardized and PCA reduces them to two dimensions for visualization. Unlike Iris, there are no ground-truth labels here, so the scatter plot shows unlabeled customers prior to clustering.

## 5.1 Basics and Algorithms
Clustering groups similar samples based on feature distances. K-means partitions data into k centroids, hierarchical clustering merges or splits nested clusters, and DBSCAN discovers density-based clusters with noise handling.

## 5.2 K-means Clustering
K-means minimizes within-cluster variance by iteratively updating centroids. It is fast and works well when clusters are spherical and balanced. Here we use 3 clusters to represent light, moderate, and heavy restaurant customers.

In [ ]:
kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
start = time.perf_counter()
kmeans_labels = kmeans.fit_predict(X_scaled)
kmeans_duration = time.perf_counter() - start
kmeans_sil = silhouette_score(X_scaled, kmeans_labels)
kmeans_db = davies_bouldin_score(X_scaled, kmeans_labels)

print('K-means runtime:', round(kmeans_duration, 4), 'seconds')
print('K-means silhouette score:', round(kmeans_sil, 4))
print('K-means Davies-Bouldin score:', round(kmeans_db, 4))

fig, ax = plt.subplots(figsize=(6, 4))
for label, color in zip([0, 1, 2], ['r', 'g', 'b']):
    ax.scatter(X_pca[kmeans_labels == label, 0], X_pca[kmeans_labels == label, 1], c=color, label=f'Cluster {label}', edgecolor='k', s=40)
ax.scatter(pca.transform(kmeans.cluster_centers_)[:, 0], pca.transform(kmeans.cluster_centers_)[:, 1], c='yellow', marker='X', s=120, edgecolor='k', label='Centroids')
ax.set_title('K-means clusters on PCA projection')
ax.legend()
plt.show()

K-means produces compact customer segments with centroid centers. The silhouette score measures cohesion and separation, while Davies-Bouldin assesses average cluster similarity.

## 5.3 Hierarchical Clustering
Hierarchical clustering builds a tree of nested clusters. Agglomerative linkage starts with each point as its own cluster and merges them step-by-step.

In [ ]:
linked = linkage(X_scaled, method='ward')
fig, ax = plt.subplots(figsize=(10, 4))
dendrogram(linked, labels=customers['Customer_ID'].astype(str).tolist(), leaf_rotation=90., leaf_font_size=9.)
ax.set_title('Hierarchical clustering dendrogram (Ward linkage)')
ax.set_xlabel('Customer ID')
ax.set_ylabel('Distance')
plt.tight_layout()
plt.show()

hier = AgglomerativeClustering(n_clusters=3, linkage='ward')
start = time.perf_counter()
hier_labels = hier.fit_predict(X_scaled)
hier_duration = time.perf_counter() - start
hier_sil = silhouette_score(X_scaled, hier_labels)
hier_db = davies_bouldin_score(X_scaled, hier_labels)

print('Hierarchical clustering runtime:', round(hier_duration, 4), 'seconds')
print('Hierarchical silhouette score:', round(hier_sil, 4))
print('Hierarchical Davies-Bouldin score:', round(hier_db, 4))

fig, ax = plt.subplots(figsize=(6, 4))
for label, color in zip([0, 1, 2], ['r', 'g', 'b']):
    ax.scatter(X_pca[hier_labels == label, 0], X_pca[hier_labels == label, 1], c=color, label=f'Cluster {label}', edgecolor='k', s=40)
ax.set_title('Hierarchical clusters on PCA projection')
ax.legend()
plt.show()

The dendrogram visualizes the hierarchical merge process across customers. Agglomerative clustering can reveal nested structure in ordering behaviour but may be slower than K-means on larger customer bases.

## 5.4 DBSCAN Clustering
DBSCAN groups points by density and marks noise. It is robust to cluster shape but sensitive to the choice of epsilon and minimum samples.

In [ ]:
dbscan = DBSCAN(eps=1.5, min_samples=3)
start = time.perf_counter()
db_labels = dbscan.fit_predict(X_scaled)
db_duration = time.perf_counter() - start
n_clusters = len(set(db_labels)) - (1 if -1 in db_labels else 0)
n_noise = list(db_labels).count(-1)
db_sil = silhouette_score(X_scaled, db_labels) if n_clusters > 1 else np.nan
db_db = davies_bouldin_score(X_scaled, db_labels) if n_clusters > 1 else np.nan

print('DBSCAN runtime:', round(db_duration, 4), 'seconds')
print('DBSCAN clusters:', n_clusters)
print('DBSCAN noise points:', n_noise)
print('DBSCAN silhouette score:', round(db_sil, 4) if not np.isnan(db_sil) else 'N/A')
print('DBSCAN Davies-Bouldin score:', round(db_db, 4) if not np.isnan(db_db) else 'N/A')

fig, ax = plt.subplots(figsize=(6, 4))
unique_labels = sorted(set(db_labels))
colors = [plt.cm.tab10(i) for i in range(len(unique_labels))]
for label, color in zip(unique_labels, colors):
    if label == -1:
        marker = 'x'
        label_name = 'Noise'
    else:
        marker = 'o'
        label_name = f'Cluster {label}'
    ax.scatter(X_pca[db_labels == label, 0], X_pca[db_labels == label, 1], c=[color], marker=marker, label=label_name, edgecolor='k', s=40)
ax.set_title('DBSCAN clusters on PCA projection')
ax.legend()
plt.show()

DBSCAN can find non-spherical clusters and detect noise (atypical customers), but metric evaluation requires at least two clusters. The epsilon and min_samples parameters were tuned for this smaller customer dataset (35 rows) compared to the Iris example.

## 5.5 Issues: Evaluation, Scalability, Comparison
Evaluation uses internal metrics such as silhouette and Davies-Bouldin scores because the customer segments have no ground-truth labels for objective comparison. K-means is scalable and fast for medium-sized data, hierarchical clustering is more computationally expensive and less scalable, while DBSCAN handles arbitrary cluster shapes but can be sensitive to parameter selection and may produce noise.

In [ ]:
comparison = pd.DataFrame([
    {'algorithm': 'K-means', 'runtime': kmeans_duration, 'silhouette': kmeans_sil, 'davies_bouldin': kmeans_db, 'clusters': len(set(kmeans_labels))},
    {'algorithm': 'Hierarchical', 'runtime': hier_duration, 'silhouette': hier_sil, 'davies_bouldin': hier_db, 'clusters': len(set(hier_labels))},
    {'algorithm': 'DBSCAN', 'runtime': db_duration, 'silhouette': db_sil, 'davies_bouldin': db_db, 'clusters': n_clusters}
])
comparison

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
comparison.plot.bar(x='algorithm', y='runtime', ax=axes[0], legend=False, color=['#4c72b0', '#dd8452', '#55a868'])
axes[0].set_title('Runtime comparison')
axes[0].set_ylabel('Seconds')
comparison.plot.bar(x='algorithm', y='silhouette', ax=axes[1], legend=False, color=['#c44e52', '#8172b2', '#2ca02c'])
axes[1].set_title('Silhouette score comparison')
comparison.plot.bar(x='algorithm', y='davies_bouldin', ax=axes[2], legend=False, color=['#8c564b', '#e377c2', '#7f7f7f'])
axes[2].set_title('Davies-Bouldin score comparison')
plt.tight_layout()
plt.show()

The comparison table and graphs summarize runtime and clustering quality across algorithms. Lower Davies-Bouldin and higher silhouette scores indicate better cluster separation and compactness among the customer segments.

In [ ]:
customers['kmeans_cluster'] = kmeans_labels
cluster_profile = customers.groupby('kmeans_cluster')[feature_names].mean().round(2)
cluster_profile['num_customers'] = customers['kmeans_cluster'].value_counts().sort_index()
cluster_profile

Profiling the K-means clusters by average feature values helps interpret the segments in business terms, for example distinguishing occasional diners from frequent, high-variety restaurant customers.

## Conclusion
The clustering lab shows that K-means gives a fast and stable customer segmentation for the restaurant orders dataset, hierarchical clustering provides a useful dendrogram for nested structure among customers at the cost of extra computation, and DBSCAN can detect noise and non-spherical structure but requires careful parameter tuning, especially on a smaller dataset of 35 customers. Overall, K-means is a strong choice for this dataset and the evaluation metrics help confirm the most consistent performer, while hierarchical and density-based clustering remain valuable for exploratory analysis and identifying atypical (noise) customers. Unlike the Iris dataset, this analysis required an additional feature-engineering step to convert raw transactional records into a numeric customer feature table before clustering could be applied.